# Port-a-Prof: QLoRA Fine-Tuning on gemma-4-E2B-it

## 1. Configuration

In [26]:
from pathlib import Path

# ── Paths ── 
# TRAINING_DIR       : directory containing the raw per-problem JSON files
# INTERMEDIATE_JSONL : all extracted examples before the train/val split
# TRAIN_JSONL        : 80 % training split
# VAL_JSONL          : 20 % validation split
TRAINING_DIR       = Path("dataset")
INTERMEDIATE_JSONL = Path("training_data_intermediate.jsonl")
TRAIN_JSONL        = Path("port_a_prof_train.jsonl")
VAL_JSONL          = Path("port_a_prof_val.jsonl")

# ── Base model ── 
MODEL_ID = "google/gemma-4-E2B-it"

assert TRAINING_DIR.exists(), f"Training directory not found: {TRAINING_DIR.resolve()}"
print("Training dir:", TRAINING_DIR.resolve())


Training dir: C:\Users\bianc\Desktop\AI-TUTOR-MODEL\dataset


## 2. Convert JSON files into chat-format training examples

Each raw JSON file contains one high-school level problem and a list of *trajectories*.
A trajectory is a sequence of tutoring-session snapshots — one per turn —
where each entry records:

- `dialogue_history` — the conversation so far (student + assistant turns)
- `internal_state`   — `current_status`, `teacher_role`, and optionally `support`
- `target_teacher_response` — the ground-truth response the model should produce

The input used for tuning is structured as:

```
## PROBLEM
<problem text>

## STUDENT_ATTEMPT
<most recent student message>

## STATUS
<misconception / support need label>

## TEACHER_ROLE
<role label>

## SUPPORT          ← only present when TEACHER_ROLE == partial_worked_step
<low/medium/high>
```



In [8]:
import json
from pathlib import Path
from typing import Any, Dict, List


def extract_latest_student_attempt(dialogue_history: List[Dict[str, str]]) -> str:
    """
    Return the most recent message from the dialouge history.
    """
    student_roles = {"student", "user"}
    for turn in reversed(dialogue_history):
        role    = (turn.get("role")    or "").strip().lower()
        content = (turn.get("content") or "").strip()
        if role in student_roles and content:
            return content
    return ""


def build_user_content(
    problem: str,
    student_attempt: str,
    internal_state: Dict[str, Any],
) -> str:
    """
    Assemble the structured user prompt from its constituent parts.

    The STATUS and TEACHER_ROLE fields encode the pedagogical context that
    the model must learn to act on.  The SUPPORT block is only
    included for the `partial_worked_step` role.
    """
    base = (
        f"## PROBLEM\n{problem.strip()}\n\n"
        f"## STUDENT_ATTEMPT\n{student_attempt.strip()}\n\n"
        f"## STATUS\n{internal_state['current_status'].strip()}\n\n"
        f"## TEACHER_ROLE\n{internal_state['teacher_role'].strip()}"
    )
    # Append the support fragment only when the role calls for it
    if internal_state['teacher_role'].strip() == "partial_worked_step":
        base += f"\n\n## SUPPORT\n{internal_state['support'].strip()}"
    return base


def extract_examples_from_problem_file(file_path: Path) -> List[Dict[str, Any]]:
    """
    Parse a single problem JSON file and return a list of training examples
    in HuggingFace chat format:

        {"messages": [{"role": "user", "content": ...},
                      {"role": "assistant", "content": ...}]}
    """
    with file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    problem     = data["problem"]
    trajectories = data["trajectories"]

    rows: List[Dict[str, Any]] = []
    for trajectory in trajectories:
        for entry in trajectory["entries"]:
            dialogue_history = entry["dialogue_history"]
            internal_state   = entry["internal_state"]
            target           = entry["target_teacher_response"].strip()

            student_attempt = extract_latest_student_attempt(dialogue_history)
            if not student_attempt:
                continue  # skip if no student turn found

            user_content = build_user_content(
                problem=problem,
                student_attempt=student_attempt,
                internal_state=internal_state,
            )

            rows.append({
                "messages": [
                    {"role": "user",      "content": user_content},
                    {"role": "assistant", "content": target},
                ]
            })
    return rows


# Iterate over every problem JSON and collect all training rows 
all_rows: List[Dict[str, Any]] = []

for json_path in sorted(TRAINING_DIR.rglob("*.json")):
    all_rows.extend(extract_examples_from_problem_file(json_path))

print(f"Extracted {len(all_rows)} total training examples.")

# Save the full intermediate set before splitting (useful for inspection)
with INTERMEDIATE_JSONL.open("w", encoding="utf-8") as f:
    for row in all_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote: {INTERMEDIATE_JSONL.resolve()}")


Extracted 462 total training examples.
Wrote: C:\Users\bianc\Desktop\AI-TUTOR-MODEL\training_data_intermediate.jsonl


## 3. Quick sanity check

In [9]:
import random

# Prints a random example 
sample = random.choice(all_rows)
print(json.dumps(sample, indent=2, ensure_ascii=False)[:4000])


{
  "messages": [
    {
      "role": "user",
      "content": "## PROBLEM\nA 25-foot ladder leans against a wall, forming an angle of 68 degrees with the ground. Determine the distance from the base of the ladder to the wall. Round your answer to the nearest foot.\n\n## STUDENT_ATTEMPT\nIf I calculate 25 times the cosine of 68 degrees, I get about 9.365. Since we have to round to the nearest foot, the distance from the base of the ladder to the wall is 9 feet.\n\n## STATUS\nThe student correctly calculates 25 times the cosine of 68 degrees, obtaining approximately 9.365. The student then correctly rounds this value to the nearest foot, concluding that the distance from the base of the ladder to the wall is 9 feet.\n\n## TEACHER_ROLE\nsession_close"
    },
    {
      "role": "assistant",
      "content": "That is correct. The distance from the base of the ladder to the wall is 9 feet. Do you have any other problems you would like to work on?"
    }
  ]
}


## 4. Train / validation split


In [10]:
from sklearn.model_selection import train_test_split

# 80 / 20 split — random_state=42 ensures the same split on every run
train_rows, val_rows = train_test_split(
    all_rows,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print(f"Train rows: {len(train_rows)}")
print(f"Val rows:   {len(val_rows)}")

# Write both splits to disk in JSONL format (one JSON object per line)
with TRAIN_JSONL.open("w", encoding="utf-8") as f:
    for row in train_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with VAL_JSONL.open("w", encoding="utf-8") as f:
    for row in val_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote: {TRAIN_JSONL.resolve()}")
print(f"Wrote: {VAL_JSONL.resolve()}")


Train rows: 369
Val rows:   93
Wrote: C:\Users\bianc\Desktop\AI-TUTOR-MODEL\port_a_prof_train.jsonl
Wrote: C:\Users\bianc\Desktop\AI-TUTOR-MODEL\port_a_prof_val.jsonl


## 5. Load dataset

Load the JSONL splits into a HuggingFace `DatasetDict` so they're compatible
with `SFTTrainer` downstream.


In [11]:
from datasets import load_dataset

# Each JSONL line is a {"messages": [...]} object — 'json' format handles this
dataset = load_dataset(
    "json",
    data_files={
        "train":      str(TRAIN_JSONL),
        "validation": str(VAL_JSONL),
    },
)

dataset


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 369
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 93
    })
})

## 6. Apply chat template

Gemma instruction-tuned models expect a specific conversation format with
special tokens that distinguish user and assistant turns.  

`enable_thinking=False` disables Gemma 4's thinking tokens so the
model generates responses directly.


In [12]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")

def formatting_prompts_func(examples):
    """
    Apply Gemma's instruction chat template to each conversation.

    tokenize=False        → return a plain string, not token IDs
    add_generation_prompt → False because the target response is included
    enable_thinking       → False to suppress chain-of-thought tokens
    """
    texts = [
            processor.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        for convo in examples["messages"]
    ]
    return {"text": texts}

# Apply the template to every split in one batched pass
dataset = dataset.map(formatting_prompts_func, batched=True)

# Sanity check — inspect one formatted prompt to confirm special tokens are present
print(dataset["train"][0]["text"])


Map:   0%|          | 0/369 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

<bos><|turn>user
## PROBLEM
A manufacturing process produces metal bolts, and 2% of them are defective. If a quality control inspector randomly selects 10 bolts, what is the probability that at least one bolt is defective?

## STUDENT_ATTEMPT
Okay, so the probability that none of the 10 bolts are defective is about 0.817. So, to find the probability of at least one being defective, I just need to do 1 minus that number.

## STATUS
The student correctly stated that the probability of no defective bolts is approximately 0.817 and accurately concluded that the probability of at least one defective bolt is found by calculating 1 minus this value.

## TEACHER_ROLE
confirm_and_advance<turn|>
<|turn>model
You have correctly found the probability of the complementary event. What value should you subtract this result from to find the probability of at least one defective bolt?<turn|>



## 7. Load gemma4-E2B-it in 4-bit for QLoRA

In [13]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())


2.8.0+cu126
12.6
True


In [14]:
from transformers import (
    AutoModelForImageTextToText,  
    BitsAndBytesConfig,
)


In [ ]:
# Quantisation config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   
    bnb_4bit_quant_type="nf4",        
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load model 
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto", 
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

# Report memory usage
print("First parameter device:", next(model.parameters()).device)
print("CUDA allocated GB:", torch.cuda.memory_allocated() / 1e9)
print("CUDA reserved  GB:", torch.cuda.memory_reserved()  / 1e9)
print("Memory footprint bytes:", model.get_memory_footprint())


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

First parameter device: cuda:0
CUDA allocated GB: 6.744275456
CUDA reserved  GB: 6.838812672
Memory footprint bytes: 6703768674


## 8. QLoRA configuration

In [16]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,                          # rank of the adapter matrices
    lora_alpha=32,                 # scaling: effective update magnitude = alpha/r * ΔW
    lora_dropout=0.0,              # no dropout 
    bias="none",                   # don't train bias terms
    target_modules="all-linear",   # attach LoRA to every linear layer in the model
    task_type="CAUSAL_LM",
    ensure_weight_tying=True,      
)

peft_config


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=True)

## 9. Training configuration

Key decisions:

- **`max_length=768`** — covers the longest prompts in the corpus without
  excessive padding on shorter ones.
- **`gradient_accumulation_steps=8`** with `per_device_train_batch_size=1`
  gives an effective batch size of 8, which is a reasonable trade-off
  between stability and VRAM on a single GPU.
- **`bf16=True`** — bfloat16 compute matches the model's storage dtype and
  avoids the loss spikes that can occur with float16 on large models.
- **Cosine LR schedule with 10 % warmup** — standard practice for
  instruction fine-tuning; prevents early destructive updates.
- **`load_best_model_at_end=True`** on `eval_loss` — automatically
  checkpoints the best epoch rather than the last.


In [ ]:
from trl import SFTConfig

args = SFTConfig(
    output_dir="port-a-prof_training",   # intermediate checkpoints go here

    # Data 
    dataset_text_field="text",           
    max_length=768,                      

    # Training schedule 
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,      

    # Optimiser
    optim="adamw_torch_fused",           
    learning_rate=1e-4,
    max_grad_norm=1.0,                   
    warmup_ratio=0.1,                    
    lr_scheduler_type="cosine",

    # Precision
    bf16=True,
    fp16=False,

    # Checkpointing & evaluation
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   
    save_total_limit=2,                

    # Reporting
    report_to="tensorboard",
    push_to_hub=False,

    # Tokenisation
    # The chat template already includes all required special tokens, so we
    # must not add them again here.
    dataset_kwargs={
        "add_special_tokens":  False,
        "append_concat_token": False,
    },
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 10. Create trainer

In [18]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,           # injects LoRA adapters into the model
    processing_class=processor.tokenizer,
)

trainer


c:\Users\bianc\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\tuners\tuners_utils.py:1209: UserWarning: You have requested `ensure_weight_tying`, but no tied modules are added in `modules_to_save`
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/369 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/369 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

## 11. Train

In [19]:
trainer.train()

# Save the best checkpoint (LoRA adapter weights + trainer state)
trainer.save_model()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.
c:\Users\bianc\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.543579,0.684301,0.672009,113917.000000,0.801054
2,0.595817,0.565465,0.522660,227834.000000,0.828589
3,0.626856,0.560188,0.463015,341751.000000,0.830340


c:\Users\bianc\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\bianc\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args

## 12. Free memory

Delete the trainer object and flush the CUDA cache before running inference,
so the GPU has enough headroom to load the adapted model for generation.


In [20]:
del trainer
torch.cuda.empty_cache()


## 13. Inference test


In [23]:
# Pick a validation example 
sample = dataset["validation"][1]

# Take only the user turn — the model must generate the assistant turn
messages = sample["messages"][:1]
print(messages)

# Apply the chat template with add_generation_prompt=True so the model knows
# to start generating immediately after the user turn
input_text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("=== INPUT PROMPT ===\n")
print(input_text)

# Tokenise and move to GPU
inputs = processor(text=input_text, return_tensors="pt")
inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

# Generate — no gradient tracking needed at inference time
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=1000)

# Slice off the prompt tokens so we only decode the generated response
prompt_len  = inputs["input_ids"].shape[1]
gen_tokens  = out[0][prompt_len:]
gen         = processor.decode(gen_tokens, skip_special_tokens=True)

print("\n=== MODEL OUTPUT ===\n")
print(gen.strip())

print("\n=== REFERENCE ===\n")
print(sample["messages"][1]["content"])


[{'role': 'user', 'content': "## PROBLEM\nA local bakery sells cookies for $2 each and muffins for $3 each. On Tuesday, they sold 120 items and earned $300. Determine the number of cookies and muffins sold using a system of elimination.\n\n## STUDENT_ATTEMPT\nI'm trying to figure out how to start this problem. We have two pieces of information—the total items and the total money—but I don't know how to turn those into equations that I can solve.\n\n## STATUS\nThe student correctly identified the two pieces of information available in the problem—the total number of items and the total money earned—but is struggling with the conceptual step of translating these real-world facts into a solvable system of algebraic equations.\n\n## TEACHER_ROLE\npartial_worked_step\n\n## SUPPORT\nlow"}]
=== INPUT PROMPT ===

<bos><|turn>user
## PROBLEM
A local bakery sells cookies for $2 each and muffins for $3 each. On Tuesday, they sold 120 items and earned $300. Determine the number of cookies and muff

## 14. Merge adapter and save final model

In [22]:
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor
import torch

# 1. Reload base model in full bfloat16 — merging requires unquantised weights
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

# 2. Attach the trained LoRA adapter (output_dir from SFTConfig)
peft_model = PeftModel.from_pretrained(base_model, args.output_dir)

# 3. Fuse adapter weights into the base model and discard the adapter scaffolding
merged_model = peft_model.merge_and_unload()

# 4. Save
merged_model.save_pretrained(
    "./port_a_prof_finetuned",
    safe_serialization=True,
    max_shard_size="2GB",
)

# 5. Save the processor alongside the model so everything needed for inference
#    lives in one directory
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
processor.save_pretrained("./port_a_prof_finetuned")


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

c:\Users\bianc\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\tuners\tuners_utils.py:1209: UserWarning: You have requested `ensure_weight_tying`, but no tied modules are added in `modules_to_save`
  warnings.warn(


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

['./port_a_prof_finetuned\\processor_config.json']